In [ ]:
import pandas as pd
from pathlib import Path


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

OUTPUT_DIR = PROJECT_ROOT / "Output"

INPUT_FILE = OUTPUT_DIR / "03_Hasil_Network_OSRM_FINAL.csv"


# ============================================================
# BACA DATA NETWORK
# ============================================================

print("Membaca hasil network...")

network = pd.read_csv(
    INPUT_FILE,
    dtype={
        "KDDESA": str,
        "kode_ppk": str
    }
)

print(
    f"Total desa network: {len(network)}"
)


# ============================================================
# NORMALISASI NAMA PROVINSI
# ============================================================

network["NMPROP"] = (
    network["NMPROP"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# FILTER MALUKU
# ============================================================

maluku = network[
    network["NMPROP"] == "MALUKU"
].copy()


# ============================================================
# FILTER MALUKU UTARA
# ============================================================

maluku_utara = network[
    network["NMPROP"] == "MALUKU UTARA"
].copy()


# ============================================================
# URUTKAN DATA
# ============================================================

sort_columns = [
    "NMDATI2",
    "NMKEC",
    "NMDESA2"
]

maluku = maluku.sort_values(
    sort_columns
).reset_index(
    drop=True
)


maluku_utara = maluku_utara.sort_values(
    sort_columns
).reset_index(
    drop=True
)


# ============================================================
# SIMPAN OUTPUT
# ============================================================

maluku_csv = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku.csv"
)

maluku_utara_csv = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku_Utara.csv"
)


maluku.to_csv(
    maluku_csv,
    index=False,
    encoding="utf-8"
)


maluku_utara.to_csv(
    maluku_utara_csv,
    index=False,
    encoding="utf-8"
)


# ============================================================
# SIMPAN EXCEL
# ============================================================

maluku_xlsx = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku.xlsx"
)

maluku_utara_xlsx = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku_Utara.xlsx"
)


maluku.to_excel(
    maluku_xlsx,
    index=False
)


maluku_utara.to_excel(
    maluku_utara_xlsx,
    index=False
)


# ============================================================
# RINGKASAN
# ============================================================

print()
print("=" * 60)
print("SELESAI")
print("=" * 60)

print()
print(f"Maluku       : {len(maluku)} desa")
print(f"Maluku Utara : {len(maluku_utara)} desa")
print(f"Total        : {len(maluku) + len(maluku_utara)} desa")
import time
import requests
import pandas as pd
from pathlib import Path


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

OUTPUT_DIR = PROJECT_ROOT / "Output"


# Existing 150-village network result
EXISTING_NETWORK_FILE = (
    OUTPUT_DIR / "15_Hasil_Network_Maluku.csv"
)


# Revised Euclidean result
# This should contain the 155 Maluku villages
REVISED_EUCLIDEAN_FILE = (
    OUTPUT_DIR / "01_Hasil_Euclidean_Desa_FKTP_Maluku_Revised.csv"
)


# Final output
FINAL_CSV = (
    OUTPUT_DIR / "15_Hasil_Network_Maluku_Revised.csv"
)

FINAL_XLSX = (
    OUTPUT_DIR / "15_Hasil_Network_Maluku_Revised.xlsx"
)


# ============================================================
# OSRM
# ============================================================

OSRM_BASE = (
    "https://router.project-osrm.org/"
    "route/v1/driving/"
)


# ============================================================
# FUNCTION: REQUEST OSRM
# ============================================================

def get_route(
    lon1,
    lat1,
    lon2,
    lat2
):

    url = (
        f"{OSRM_BASE}"
        f"{lon1},{lat1};"
        f"{lon2},{lat2}"
        f"?overview=full"
        f"&geometries=geojson"
    )

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# 1. BACA DATA
# ============================================================

print("=" * 70)
print("MEMBACA DATA")
print("=" * 70)


existing = pd.read_csv(
    EXISTING_NETWORK_FILE,
    dtype={
        "KDDESA": str,
        "kode_ppk": str
    }
)


revised_euclidean = pd.read_csv(
    REVISED_EUCLIDEAN_FILE,
    dtype={
        "KDDESA": str,
        "kode_ppk": str
    }
)


print(
    f"Network Maluku lama : {len(existing)} desa"
)


print(
    f"Euclidean revisi    : {len(revised_euclidean)} desa"
)


# ============================================================
# 2. NORMALISASI KODE DESA
# ============================================================

existing["KDDESA"] = (
    existing["KDDESA"]
    .astype(str)
    .str.strip()
)


revised_euclidean["KDDESA"] = (
    revised_euclidean["KDDESA"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 3. CARI DESA BARU
# ============================================================

existing_codes = set(
    existing["KDDESA"]
)


new_villages = revised_euclidean[
    ~revised_euclidean["KDDESA"]
    .isin(existing_codes)
].copy()


print()
print("=" * 70)
print("DESA BARU YANG BELUM ADA DI NETWORK LAMA")
print("=" * 70)


print(
    f"Jumlah desa baru: {len(new_villages)}"
)


if len(new_villages) > 0:

    print()

    print(
        new_villages[
            [
                "KDDESA",
                "NMDESA2",
                "NMKEC",
                "NMDATI2"
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 4. HITUNG NETWORK UNTUK DESA BARU
# ============================================================

new_network_rows = []


for number, (_, row) in enumerate(
    new_villages.iterrows(),
    start=1
):

    print()
    print(
        f"{number}/{len(new_villages)} "
        f"{row['NMDESA2']} -> REQUEST OSRM"
    )


    result = None


    for attempt in range(
        1,
        4
    ):

        try:

            route_result = get_route(

                row["Longitude_Desa"],

                row["Latitude_Desa"],

                row["Longitude_FKTP"],

                row["Latitude_FKTP"]

            )


            if route_result.get(
                "code"
            ) == "Ok":

                route_data = (
                    route_result[
                        "routes"
                    ][0]
                )


                network_meter = round(
                    route_data[
                        "distance"
                    ],
                    2
                )


                network_km = round(
                    network_meter / 1000,
                    3
                )


                duration_minute = round(
                    route_data[
                        "duration"
                    ] / 60,
                    2
                )


                status = (
                    "ACCESS"
                )


            else:

                network_meter = None

                network_km = None

                duration_minute = None

                status = (
                    "NO ROAD ACCESS"
                )


            result = {

                "KDDESA":
                    row["KDDESA"],

                "NMDESA2":
                    row["NMDESA2"],

                "NMKEC":
                    row["NMKEC"],

                "NMDATI2":
                    row["NMDATI2"],

                "NMPROP":
                    row["NMPROP"],

                "Longitude_Desa":
                    row["Longitude_Desa"],

                "Latitude_Desa":
                    row["Latitude_Desa"],

                "kode_ppk":
                    row["kode_ppk"],

                "nmppk":
                    row["nmppk"],

                "Longitude_FKTP":
                    row["Longitude_FKTP"],

                "Latitude_FKTP":
                    row["Latitude_FKTP"],

                "Jarak_Meter":
                    row["Jarak_Meter"],

                "Jarak_KM":
                    row["Jarak_KM"],

                "Network_Meter":
                    network_meter,

                "Network_KM":
                    network_km,

                "Duration_Minute":
                    duration_minute,

                "Status":
                    status

            }


            break


        except Exception as error:

            print(
                f"   Retry {attempt}/3"
            )

            print(
                f"   Error: {error}"
            )


            if attempt < 3:

                time.sleep(
                    3
                )


    if result is None:

        result = {

            "KDDESA":
                row["KDDESA"],

            "NMDESA2":
                row["NMDESA2"],

            "NMKEC":
                row["NMKEC"],

            "NMDATI2":
                row["NMDATI2"],

            "NMPROP":
                row["NMPROP"],

            "Longitude_Desa":
                row["Longitude_Desa"],

            "Latitude_Desa":
                row["Latitude_Desa"],

            "kode_ppk":
                row["kode_ppk"],

            "nmppk":
                row["nmppk"],

            "Longitude_FKTP":
                row["Longitude_FKTP"],

            "Latitude_FKTP":
                row["Latitude_FKTP"],

            "Jarak_Meter":
                row["Jarak_Meter"],

            "Jarak_KM":
                row["Jarak_KM"],

            "Network_Meter":
                None,

            "Network_KM":
                None,

            "Duration_Minute":
                None,

            "Status":
                "SERVER ERROR"

        }


    new_network_rows.append(
        result
    )


    time.sleep(
        1
    )


# ============================================================
# 5. GABUNG DATA LAMA + DATA BARU
# ============================================================

new_network = pd.DataFrame(
    new_network_rows
)


final_network = pd.concat(
    [
        existing,
        new_network
    ],
    ignore_index=True
)


# ============================================================
# 6. HAPUS DUPLIKAT
# ============================================================

final_network = (
    final_network
    .drop_duplicates(
        subset=[
            "KDDESA"
        ],
        keep="last"
    )
)


# ============================================================
# 7. URUTKAN
# ============================================================

final_network = (
    final_network
    .sort_values(
        [
            "NMDATI2",
            "NMKEC",
            "NMDESA2"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. SIMPAN
# ============================================================

final_network.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8"
)


final_network.to_excel(
    FINAL_XLSX,
    index=False
)


# ============================================================
# 9. RINGKASAN
# ============================================================

print()
print("=" * 70)
print("SELESAI")
print("=" * 70)

print()
print(
    f"Network lama : {len(existing)} desa"
)

print(
    f"Desa baru    : {len(new_network)} desa"
)

print(
    f"Total final  : {len(final_network)} desa"
)

print()

print(
    "STATUS:"
)

print(
    final_network[
        "Status"
    ].value_counts()
)


print()
print(
    "CSV tersimpan di:"
)

print(
    FINAL_CSV
)


print()
print(
    "Excel tersimpan di:"
)

print(
    FINAL_XLSX
)
print()
print("CSV Maluku:")
print(maluku_csv)

print()
print("CSV Maluku Utara:")
print(maluku_utara_csv)

print()
print("Excel Maluku:")
print(maluku_xlsx)

print()
print("Excel Maluku Utara:")
print(maluku_utara_xlsx)

In [ ]:
import time
from pathlib import Path

import requests
import pandas as pd


# ============================================================
# PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent  # run from the Script/ folder

OUTPUT_DIR = PROJECT_ROOT / "Output"


# ============================================================
# INPUT
# ============================================================

EXISTING_NETWORK_FILE = OUTPUT_DIR / (
    "03_Hasil_Network_OSRM_FINAL.xlsx"
)

REVISED_EUCLIDEAN_FILE = OUTPUT_DIR / (
    "01_Hasil_Euclidean_Maluku_Revised.csv"
)


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_XLSX = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku_Revised.xlsx"
)

OUTPUT_CSV = OUTPUT_DIR / (
    "15_Hasil_Network_Maluku_Revised.csv"
)


# ============================================================
# OSRM
# ============================================================

BASE_URL = (
    "https://router.project-osrm.org/route/v1/driving/"
)


# ============================================================
# 5 NEW VILLAGES
# ============================================================

NEW_VILLAGES = [

    "BATU KARANG",

    "AKAT FADEDO",

    "KWAWOR BESAR ENA",

    "KWAWOR KECIL MATA ATA",

    "URUNG"

]


# ============================================================
# OSRM REQUEST
# ============================================================

def get_route(

    lon1,

    lat1,

    lon2,

    lat2

):

    url = (

        f"{BASE_URL}"

        f"{lon1},{lat1};"

        f"{lon2},{lat2}"

        "?overview=full"

        "&geometries=geojson"

    )


    response = requests.get(

        url,

        timeout=60

    )


    response.raise_for_status()


    return response.json()


# ============================================================
# MAIN
# ============================================================

print("=" * 70)

print("OSRM ROUTING - 5 NEW MALUKU DESA")

print("=" * 70)


# ============================================================
# 1. READ EXISTING NETWORK
# ============================================================

print("\nMembaca network lama...")


network = pd.read_excel(

    EXISTING_NETWORK_FILE,

    dtype={

        "KDDESA": str,

        "kode_ppk": str

    }

)


print(

    f"Network lama: {len(network)} desa"

)


# ============================================================
# 2. READ REVISED EUCLIDEAN
# ============================================================

print("\nMembaca Euclidean revisi...")


euclidean = pd.read_csv(

    REVISED_EUCLIDEAN_FILE,

    dtype={

        "KDDESA": str,

        "kode_ppk": str

    }

)


# ============================================================
# 3. SELECT ONLY THE 5 NEW DESA
# ============================================================

new_data = euclidean[

    euclidean["NMDESA2"]

    .astype(str)

    .str.upper()

    .str.strip()

    .isin(NEW_VILLAGES)

].copy()


print(

    f"Desa baru ditemukan: {len(new_data)}"

)


if len(new_data) == 0:

    raise ValueError(

        "Tidak ada desa baru yang ditemukan."

    )


print(

    new_data[

        [

            "KDDESA",

            "NMDESA2",

            "NMKEC",

            "NMDATI2"

        ]

    ].to_string(

        index=False

    )

)


# ============================================================
# 4. PROCESS OSRM ONLY FOR THE 5 NEW DESA
# ============================================================

new_results = []


for position, (_, row) in enumerate(

    new_data.iterrows(),

    start=1

):


    print()

    print(

        f"{position}/{len(new_data)} "

        f"{row['NMDESA2']} "

        "-> REQUEST OSRM"

    )


    record = row.to_dict()


    success = False


    for attempt in range(1, 4):


        try:


            result = get_route(

                row["Longitude_Desa"],

                row["Latitude_Desa"],

                row["Longitude_FKTP"],

                row["Latitude_FKTP"]

            )


            # ------------------------------------------------
            # ROUTE FOUND
            # ------------------------------------------------

            if result.get("code") == "Ok":


                route = result["routes"][0]


                network_meter = (

                    route["distance"]

                )


                network_km = (

                    network_meter / 1000

                )


                duration_minute = (

                    route["duration"] / 60

                )


                record["Network_Meter"] = round(

                    network_meter,

                    2

                )


                record["Network_KM"] = round(

                    network_km,

                    3

                )


                record["Duration_Minute"] = round(

                    duration_minute,

                    2

                )


                record["Status"] = (

                    "ACCESS"

                )


                print(

                    f"   ACCESS"

                )


            # ------------------------------------------------
            # NO ROAD
            # ------------------------------------------------

            else:


                record["Network_Meter"] = None

                record["Network_KM"] = None

                record["Duration_Minute"] = None

                record["Status"] = (

                    "NO ROAD ACCESS"

                )


                print(

                    f"   NO ROAD ACCESS"

                )


            success = True


            break


        except Exception as error:


            print(

                f"   Retry {attempt}/3"

            )


            print(

                f"   Error: {error}"

            )


            time.sleep(3)


    # --------------------------------------------------------
    # SERVER ERROR AFTER 3 ATTEMPTS
    # --------------------------------------------------------

    if not success:


        record["Network_Meter"] = None

        record["Network_KM"] = None

        record["Duration_Minute"] = None

        record["Status"] = (

            "SERVER ERROR"

        )


        print(

            "   SERVER ERROR"

        )


    new_results.append(

        record

    )


    time.sleep(1)


# ============================================================
# 5. CREATE NEW DATAFRAME
# ============================================================

new_network = pd.DataFrame(

    new_results

)


# ============================================================
# 6. REMOVE THE 5 OLD VERSIONS FROM NETWORK DATA
# ============================================================

new_keys = set(

    new_network["KDDESA"]

    .astype(str)

    .str.strip()

)


network = network[

    ~network["KDDESA"]

    .astype(str)

    .str.strip()

    .isin(new_keys)

].copy()


# ============================================================
# 7. COMBINE OLD NETWORK + NEW OSRM RESULTS
# ============================================================

final = pd.concat(

    [

        network,

        new_network

    ],

    ignore_index=True

)


# ============================================================
# 8. SORT
# ============================================================

final = final.sort_values(

    [

        "NMPROP",

        "NMDATI2",

        "NMKEC",

        "NMDESA2"

    ],

    na_position="last"

)


final = final.reset_index(

    drop=True

)


# ============================================================
# 9. SAVE
# ============================================================

final.to_excel(

    OUTPUT_XLSX,

    index=False

)


final.to_csv(

    OUTPUT_CSV,

    index=False,

    encoding="utf-8"

)


# ============================================================
# 10. SUMMARY
# ============================================================

print()

print("=" * 70)

print("SELESAI")

print("=" * 70)


print()

print(

    "Hasil 5 desa baru:"

)


print(

    new_network[

        [

            "KDDESA",

            "NMDESA2",

            "Network_Meter",

            "Network_KM",

            "Duration_Minute",

            "Status"

        ]

    ].to_string(

        index=False

    )

)


print()

print(

    f"Total data Maluku akhir: {len(final)} desa"

)


print()

print(

    "Status keseluruhan:"

)


print(

    final["Status"]

    .value_counts(

        dropna=False

    )

)


print()

print(

    "Output Excel:"

)


print(

    OUTPUT_XLSX

)


print()

print(

    "Output CSV:"

)


print(

    OUTPUT_CSV

)